In [1]:
from pathlib import Path
import pandas as pd

project_root = Path("/home/dwp46550/ba_nylon")

df = pd.read_csv(
    project_root / "matrices" / "NylC_Puetz_raw_data.CSV",
    sep=";",
    decimal=","
)
print(df.shape)

(36, 7)


In [2]:
from pathlib import Path
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

project_root = Path("/home/dwp46550/ba_nylon")
wt_pdb = project_root / "data" / "raw" / "wt.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("WT", wt_pdb)
model = structure[0]

target_positions = [99, 134, 301, 304, 330]

n_terminal_chains = ["A", "C", "E", "G"]
c_terminal_chains = ["B", "D", "F", "H"]

print("Available chains:")
print([chain.id for chain in model])

print("\nChain overview:")
for chain in model:
    residues = [res for res in chain if res.id[0] == " "]

    seq = "".join(
        seq1(res.resname)
        for res in residues
    )

    print(
        f"Chain {chain.id}: "
        f"{len(residues)} residues, "
        f"residue IDs {residues[0].id[1]}-{residues[-1].id[1]}, "
        f"sequence length {len(seq)}"
    )

print("\nTarget residue mapping:")
for chain_id in n_terminal_chains + c_terminal_chains:
    chain = model[chain_id]

    for pos in target_positions:
        key = (" ", pos, " ")

        if key in chain:
            print(
                f"Chain {chain_id}, position {pos}: "
                f"{chain[key].resname}"
            )

print("\nValidating expected NylC mutation positions:")

expected_residues = {
    "A": {99: "ASP", 134: "PHE"},
    "C": {99: "ASP", 134: "PHE"},
    "E": {99: "ASP", 134: "PHE"},
    "G": {99: "ASP", 134: "PHE"},
    "B": {301: "PHE", 304: "ASP", 330: "ARG"},
    "D": {301: "PHE", 304: "ASP", 330: "ARG"},
    "F": {301: "PHE", 304: "ASP", 330: "ARG"},
    "H": {301: "PHE", 304: "ASP", 330: "ARG"},
}

for chain_id, positions in expected_residues.items():
    chain = model[chain_id]

    for pos, expected_resname in positions.items():
        key = (" ", pos, " ")

        if key not in chain:
            raise ValueError(
                f"Missing residue {pos} in chain {chain_id}"
            )

        observed_resname = chain[key].resname

        if observed_resname != expected_resname:
            raise ValueError(
                f"Unexpected residue at chain {chain_id}, position {pos}: "
                f"observed {observed_resname}, expected {expected_resname}"
            )

print("All expected mutation positions are valid.")

print("\nReference sequence fragments:")
for chain_id in ["A", "B"]:
    chain = model[chain_id]
    residues = [res for res in chain if res.id[0] == " "]

    seq = "".join(
        seq1(res.resname)
        for res in residues
    )

    print(f">{chain_id}")
    print(seq)

Available chains:
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

Chain overview:
Chain A: 243 residues, residue IDs 18-260, sequence length 243
Chain B: 89 residues, residue IDs 267-355, sequence length 89
Chain C: 247 residues, residue IDs 15-261, sequence length 247
Chain D: 89 residues, residue IDs 267-355, sequence length 89
Chain E: 243 residues, residue IDs 18-260, sequence length 243
Chain F: 89 residues, residue IDs 267-355, sequence length 89
Chain G: 247 residues, residue IDs 15-261, sequence length 247
Chain H: 89 residues, residue IDs 267-355, sequence length 89

Target residue mapping:
Chain A, position 99: ASP
Chain A, position 134: PHE
Chain C, position 99: ASP
Chain C, position 134: PHE
Chain E, position 99: ASP
Chain E, position 134: PHE
Chain G, position 99: ASP
Chain G, position 134: PHE
Chain B, position 301: PHE
Chain B, position 304: ASP
Chain B, position 330: ARG
Chain D, position 301: PHE
Chain D, position 304: ASP
Chain D, position 330: ARG
Chain F, position 301: PH

In [ ]:
# Determine the BoltzGen indices of the redesignable NylC positions

boltzgen_target_positions = [
    {"chain_id": "A", "nylc_position": 99,  "expected_residue": "ASP"},
    {"chain_id": "A", "nylc_position": 134, "expected_residue": "PHE"},
    {"chain_id": "D", "nylc_position": 304, "expected_residue": "ASP"},
    {"chain_id": "H", "nylc_position": 330, "expected_residue": "ARG"},
]

# BoltzGen numbers the exported chain sequences according to their order
# in the input structure: A -> full_sequence_0, ..., H -> full_sequence_7.
chain_to_sequence_column = {
    chain.id: f"full_sequence_{index}"
    for index, chain in enumerate(model)
}


def determine_local_residue_index(chain, pdb_position):
    """Return the one-based sequence index and residue for a PDB position."""
    residues = [
        residue
        for residue in chain
        if residue.id[0] == " "
    ]

    matches = [
        (local_index, residue)
        for local_index, residue in enumerate(residues, start=1)
        if residue.id[1] == pdb_position
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one residue at PDB position {pdb_position} "
            f"in chain {chain.id}, but found {len(matches)}."
        )

    return matches[0]


BOLTZGEN_POSITION_MAP = []

print("BoltzGen position mapping:")

for target in boltzgen_target_positions:
    chain_id = target["chain_id"]
    nylc_position = target["nylc_position"]
    expected_residue = target["expected_residue"]
    chain = model[chain_id]

    local_index, residue = determine_local_residue_index(
        chain=chain,
        pdb_position=nylc_position,
    )

    if residue.resname != expected_residue:
        raise ValueError(
            f"Unexpected residue at chain {chain_id}, position {nylc_position}: "
            f"observed {residue.resname}, expected {expected_residue}."
        )

    sequence_column = chain_to_sequence_column[chain_id]
    BOLTZGEN_POSITION_MAP.append(
        (sequence_column, local_index, nylc_position)
    )

    print(
        f"NylC position {nylc_position}: chain {chain_id}, "
        f"PDB residue {residue.resname}, BoltzGen res_index {local_index}, "
        f"output column {sequence_column}"
    )

print("\nBOLTZGEN_POSITION_MAP = [")
for entry in BOLTZGEN_POSITION_MAP:
    print(f"    {entry},")
print("]")


sequence from suplementary puetz et al

In [3]:
from Bio.Seq import Seq

target_positions = [99, 134, 301, 304, 330]
expected_residues = ["D", "F", "F", "D", "R"]

wt_nucleotide_sequence = """
ATGATGCATCATCATCATCATCACGGGGCAGGTGCCAATACCACACCGGTTCATGCACT
GACCGATATTGATGGTGGTATTGCAGTTGATCCGGCACCGCGTCTGGCAGGTCCGCCT
GTTTTTGGTGGTCCGGGTAATGCTGCATTCGATCTGGCACCGGTTCGTAGCACCGGTC
GTGAAATGCTGCGTTTTGATTTTCCGGGTGTTAGCATTGGTGCAGCACATTATGAAGAA
GGTCCGACAGGCGCAACCGTTATTCATATTCCGGCAGGCGCACGTACCGCAGTTGATG
CACGTGGTGGTGCAGTTGGTCTGAGCGGTGGTTATGATTTTAATCATGCAATTTGCCTG
GCAGGCGGTGCAGGTTATGGTCTGGAAGCCGGTGCCGGTGTTAGTGGTGCACTGCTG
GAACGTCTGGAATATCGTACCGGTTTTGCAGAACTGCAGCTGGTTAGCAGCGCAGTTAT
CTATGATTTTTCAGCACGTTCAACCGCAGTTTATCCTGATAAAGCACTGGGTCGTGCAG
CACTGGAATTTGCAGTTCCGGGTGAATTTCCGCAGGGTCGTGCCGGTGCGGGTATGAG
CGCAAGCGCAGGTAAAGTTGATTGGGATCGTACCGAAATTACCGGTCAGGGTGCAGCC
TTTCGTCGTCTGGGTGATGTTCGTATTCTGGCAGTTGTTGTTCCGAATCCGGTTGGTGT
TATTGTTGATCGTGCAGGCACCGTTGTTCGTGGTAATTATGATGCACAGACCGGTGTTC
GTCGTCATCCGGTTTTTGATTATCAAGAAGCATTTGCCGAACAGGTTCCTCCGGTTACC
CAAGCAGGTAATACCACAATTAGCGCCATTGTTACCAATGTGCGTATGAGTCCGGTTGA
ACTGAATCAGTTTGCGAAACAGGTTCATAGCAGCATGCATCGTGGCATTCAGCCGTTTC
ATACAGATATGGATGGTGATACCCTGTTTGCAGTTACCACCGATGAAATTGATCTGCCG
ACAACACCGGGTAGCAGCCGTGGTCGTCTGAGCGTTAATGCAACCGCACTGGGTGCAA
TTGCCAGCGAAGTTATGTGGGATGCCGTTCTGGAAGCGGGTAAATAA
"""

wt_nucleotide_sequence = (
    wt_nucleotide_sequence
    .replace("\n", "")
    .replace(" ", "")
    .upper()
)

wt_amino_sequence = str(
    Seq(wt_nucleotide_sequence).translate(to_stop=True)
)

print("Translated sequence:")
print(wt_amino_sequence)
print("Length with tag:", len(wt_amino_sequence))

print("\nChecking offsets:")
best_offset = None

for offset in range(0, 30):
    candidate = wt_amino_sequence[offset:]

    if len(candidate) < max(target_positions):
        continue

    observed = [
        candidate[pos - 1]
        for pos in target_positions
    ]

    print(offset, len(candidate), observed)

    if observed == expected_residues:
        best_offset = offset

if best_offset is None:
    raise ValueError(
        "No offset produced the expected NylC residues "
        f"{dict(zip(target_positions, expected_residues))}"
    )

wt_reference = wt_amino_sequence[best_offset:]

print("\nSelected reference:")
print("Offset:", best_offset)
print("Length:", len(wt_reference))
print(wt_reference)

print("\nFinal validation:")
for pos, expected in zip(target_positions, expected_residues):
    observed = wt_reference[pos - 1]
    print(pos, observed)

    assert observed == expected, (
        f"Position {pos}: expected {expected}, found {observed}"
    )

print("\nReference sequence is valid.")

Translated sequence:
MMHHHHHHGAGANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRSTGREMLRFDFPGVSIGAAHYEEGPTGATVIHIPAGARTAVDARGGAVGLSGGYDFNHAICLAGGAGYGLEAGAGVSGALLERLEYRTGFAELQLVSSAVIYDFSARSTAVYPDKALGRAALEFAVPGEFPQGRAGAGMSASAGKVDWDRTEITGQGAAFRRLGDVRILAVVVPNPVGVIVDRAGTVVRGNYDAQTGVRRHPVFDYQEAFAEQVPPVTQAGNTTISAIVTNVRMSPVELNQFAKQVHSSMHRGIQPFHTDMDGDTLFAVTTDEIDLPTTPGSSRGRLSVNATALGAIASEVMWDAVLEAGK
Length with tag: 366

Checking offsets:
0 366 ['R', 'A', 'V', 'S', 'D']
1 365 ['G', 'L', 'H', 'M', 'L']
2 364 ['G', 'L', 'S', 'H', 'P']
3 363 ['A', 'E', 'S', 'R', 'T']
4 362 ['V', 'R', 'M', 'G', 'T']
5 361 ['G', 'L', 'H', 'I', 'P']
6 360 ['L', 'E', 'R', 'Q', 'G']
7 359 ['S', 'Y', 'G', 'P', 'S']
8 358 ['G', 'R', 'I', 'F', 'S']
9 357 ['G', 'T', 'Q', 'H', 'R']
10 356 ['Y', 'G', 'P', 'T', 'G']
11 355 ['D', 'F', 'F', 'D', 'R']
12 354 ['F', 'A', 'H', 'M', 'L']
13 353 ['N', 'E', 'T', 'D', 'S']
14 352 ['H', 'L', 'D', 'G', 'V']
15 351 ['A', 'Q', 'M', 'D', 'N']
16 350 ['I', 'L', 'D', 'T', 'A']
17 349 ['C', 'V', 'G'

In [4]:
from pathlib import Path
import pandas as pd

project_root = Path("/home/dwp46550/ba_nylon")

df = pd.read_csv(
    project_root / "matrices" / "NylC_Puetz_raw_data.CSV",
    sep=";",
    decimal=","
)

# remove His-tag
wt_reference = wt_amino_sequence[11:]

assert len(wt_reference) == 355
assert wt_reference[98] == "D"
assert wt_reference[133] == "F"
assert wt_reference[300] == "F"
assert wt_reference[303] == "D"
assert wt_reference[329] == "R"

out_dir = project_root / "inputs" / "variant_fastas"
out_dir.mkdir(parents=True, exist_ok=True)

def apply_mutations(seq, mutations):
    seq = list(seq)

    if pd.isna(mutations) or str(mutations).strip() == "":
        return "".join(seq)

    mutations = str(mutations).replace(":", ";")

    for mutation in mutations.split(";"):
        mutation = mutation.strip()

        wt_aa = mutation[0]
        pos = int(mutation[1:-1])
        new_aa = mutation[-1]

        idx = pos - 1

        if seq[idx] != wt_aa:
            raise ValueError(
                f"{mutation} does not match reference sequence. "
                f"Position {pos}: expected {wt_aa}, found {seq[idx]}"
            )

        seq[idx] = new_aa

    return "".join(seq)

for _, row in df.iterrows():
    variant_id = row["variant_id"]
    mutations = row["mutations"]

    variant_seq = apply_mutations(wt_reference, mutations)

    fasta_path = out_dir / f"{variant_id}.fasta"

    with open(fasta_path, "w") as f:
        # Boltz-compatible FASTA header
        f.write(">A|protein\n")
        f.write(variant_seq + "\n")

print(f"Written {len(df)} FASTA files to {out_dir}")

Written 36 FASTA files to /home/dwp46550/ba_nylon/inputs/variant_fastas


In [7]:
from Bio import SeqIO

fasta_dir = out_dir      
reference_file = out_dir/ "WT.fasta"

reference_record = next(SeqIO.parse(reference_file, "fasta"))
reference_seq = str(reference_record.seq)

rows = []

for fasta_file in fasta_dir.glob("*.fasta"):

    record = next(SeqIO.parse(fasta_file, "fasta"))
    seq = str(record.seq)

    mutations = []

    for pos, (wt_aa, var_aa) in enumerate(
        zip(reference_seq, seq),
        start=1
    ):
        if wt_aa != var_aa:
            mutations.append(f"{wt_aa}{pos}{var_aa}")

    rows.append({
        "variant": fasta_file.stem,
        "n_mut": len(mutations),
        "mutations": ";".join(mutations),
        "sequence": seq
    })

mutation_df = pd.DataFrame(rows)

print(f"Loaded {len(mutation_df)} variants")
display(mutation_df.head(36))

Loaded 36 variants


,variant,n_mut,mutations,sequence
0,D304W,1,D304W,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
1,D304Q,1,D304Q,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
2,F134W_D304L,2,F134W;D304L,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
3,R330Q,1,R330Q,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
4,D99V,1,D99V,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
5,D304E,1,D304E,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
6,WT,0,,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
7,F134W_D304R,2,F134W;D304R,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
8,D99V_F134W_D304M_R330A,4,D99V;F134W;D304M;R330A,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...
9,D304V,1,D304V,ANTTPVHALTDIDGGIAVDPAPRLAGPPVFGGPGNAAFDLAPVRST...


In [9]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = mutation_df.copy()

# -----------------------------
# Mutation lists
# -----------------------------
df["mutation_list"] = (
    df["mutations"]
    .fillna("")
    .apply(lambda x: x.split(";") if x else [])
)

# -----------------------------
# Single mutation one-hot
# e.g. D99R, F134W, D304M
# -----------------------------
mlb_mut = MultiLabelBinarizer()

mutation_onehot = pd.DataFrame(
    mlb_mut.fit_transform(df["mutation_list"]),
    columns=[f"mut_{m}" for m in mlb_mut.classes_],
    index=df.index
)

# -----------------------------
# Mutated position one-hot
# e.g. pos_99, pos_134, pos_304
# -----------------------------
def extract_position(mutation):
    return "".join([c for c in mutation if c.isdigit()])

df["position_list"] = df["mutation_list"].apply(
    lambda muts: [extract_position(m) for m in muts]
)

mlb_pos = MultiLabelBinarizer()

position_onehot = pd.DataFrame(
    mlb_pos.fit_transform(df["position_list"]),
    columns=[f"pos_{p}" for p in mlb_pos.classes_],
    index=df.index
)

# -----------------------------
# Epistasis / interaction flags
# -----------------------------
df["has_D99"] = df["position_list"].apply(lambda x: int("99" in x))
df["has_F134W"] = df["mutation_list"].apply(lambda x: int("F134W" in x))
df["has_D304"] = df["position_list"].apply(lambda x: int("304" in x))
df["has_R330"] = df["position_list"].apply(lambda x: int("330" in x))

df["has_D99R"] = df["mutation_list"].apply(lambda x: int("D99R" in x))
df["has_D304M"] = df["mutation_list"].apply(lambda x: int("D304M" in x))
df["has_R330A"] = df["mutation_list"].apply(lambda x: int("R330A" in x))

df["epistasis_D99R_D304"] = (
    (df["has_D99R"] == 1) &
    (df["has_D304"] == 1)
).astype(int)

df["hp_like_core"] = (
    (df["has_F134W"] == 1) &
    (df["has_D304M"] == 1) &
    (df["has_R330A"] == 1)
).astype(int)

# -----------------------------
# Final feature table
# -----------------------------
base_features = df[
    [
        "variant",
        "n_mut",
        "has_D99",
        "has_F134W",
        "has_D304",
        "has_R330",
        "has_D99R",
        "has_D304M",
        "has_R330A",
        "epistasis_D99R_D304",
        "hp_like_core",
    ]
]

feature_df = pd.concat(
    [
        base_features,
        mutation_onehot,
        position_onehot
    ],
    axis=1
)

display(feature_df.head(36))
print(feature_df.shape)

,variant,n_mut,has_D99,has_F134W,has_D304,has_R330,has_D99R,has_D304M,has_R330A,epistasis_D99R_D304,...,mut_D99V,mut_F134W,mut_F301L,mut_R330A,mut_R330Q,pos_134,pos_301,pos_304,pos_330,pos_99
0,D304W,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,D304Q,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,F134W_D304L,2,0,1,1,0,0,0,0,0,...,0,1,0,0,0,1,0,1,0,0
3,R330Q,1,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
4,D99V,1,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
5,D304E,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
6,WT,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,F134W_D304R,2,0,1,1,0,0,0,0,0,...,0,1,0,0,0,1,0,1,0,0
8,D99V_F134W_D304M_R330A,4,1,1,1,1,0,1,1,0,...,1,1,0,1,0,1,0,1,1,1
9,D304V,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


(36, 32)
